In [136]:
from music21 import converter, metadata, key
import pandas as pd

def beat_to_frac(beat_in_measure, timesig):
    beats_per_measure = timesig.numerator
    beat_fraction = (beat_in_measure - 1)/ beats_per_measure
    return beat_fraction
    
def get_notes(rel_paths, fnames):
    
    base_path = "/Users/Jessica/Documents/ERIP2025/Jessica-ERIP2025/jazz_transcriptions"
    xml_path =  base_path + "/" + rel_paths + "/" + fnames + ".xml"
    # === Load your MusicXML file ===
    score = converter.parse(xml_path)  # <-- replace with your filename
    
    # === Prepare rows ===
    rows = []
    
    # === Iterate over parts ===
    for part in score.parts:
        part_name = part.partName if part.partName else 'Unknown Part'
        
        # Iterate over all notes/chords in this part
        for n in part.recurse().notes:
            measure_num = n.measureNumber  # SAFE and consistent
            beat_in_measure = n.beat
            timesig = n.getContextByClass('TimeSignature')
            beat_frac = beat_to_frac(beat_in_measure, timesig)
            abs_time = measure_num + beat_frac
            duration = n.quarterLength 
            
            # Handle Note
            if n.isNote:
                row = {
                    'order': abs_time,
                    'measure': measure_num,
                    'beat': beat_in_measure,
                    'part': part_name,
                    'midi': n.pitch.midi,
                    'pc': n.pitch.pitchClass,
                    'note': n.name,
                    'duration': duration
                }
                rows.append(row)
            
            # Handle Chord (multiple pitches)
            elif n.isChord:
                for p in n.pitches:
                    row = {
                        'order': abs_time,
                        'measure': measure_num,
                        'beat': beat_in_measure,
                        'part': part_name,
                        'midi': p.midi,
                        'pc': p.pitchClass,
                        'note': p.name,
                        'duration': duration
                    }
                    rows.append(row)
    
    # === Build DataFrame ===
    df = pd.DataFrame(rows)
    df = df.sort_values(by=['order', 'part', 'midi']).reset_index(drop=True)
    
    # === Extract metadata ===
    md = score.metadata
    artist = md.composer if md and md.composer else 'Unknown'
    title = md.title if md and md.title else 'Unknown'
    year = md.date if md and md.date else 'Unknown'
    
    # === Extract key ===
    key_sig = None
    for ks in score.recurse().getElementsByClass('KeySignature'):
        key_sig = ks
        break
    
    if key_sig:
        detected_key = key_sig.asKey().name
    else:
        detected_key = score.analyze('key').name
    
    # === Add metadata to df.attrs ===
    df.attrs['artist'] = artist
    df.attrs['title'] = title
    df.attrs['year'] = year
    df.attrs['key'] = detected_key
    df.attrs['rel_paths'] = rel_paths
    df.attrs['fnames'] = fnames
    
    return df

# Takes df and returns df with columns 'note' and 'pc' with all notes occurring at the same time as list
def generate_pcs(df_raw):
    pitchclass = df_raw.copy()
    pitchclass = pitchclass.groupby(by=['order'])[['note', 'pc']].agg(list).reset_index()
    pitchclass['pcs'] = pitchclass['pc'].apply(lambda x: list(set(x)))
    return pitchclass


In [138]:
from fractions import Fraction
pitch_class_names = ['C', 'C#/Db', 'D', 'D#/Eb', 'E', 'F',
                   'F#/Gb', 'G', 'G#/Ab', 'A', 'A#/Bb', 'B']
def preprocess_df(fname):
    base_path = "/Users/Jessica/Documents/ERIP2025/Jessica-ERIP2025/jazz_transcriptions/notes" 
    path_row = base_path + "/" + fname + ".tsv"

    try: 
        df = pd.read_csv(path_row, sep='\t')
        df['pc'] = df['midi'] % 12
        df = df[df['pc'].notna()]
        df['pc'] = df['pc'].apply(int)
        df['note'] = df['pc'].map(lambda x: pitch_class_names[x])
        
        df['mc_onset_float'] = df['mc_onset'].apply(lambda x: float(Fraction(x)))
        df['order'] = df['mc'] + df['mc_onset_float']

    except Exception as e:
        print(f"{e}")
    
    return df

In [140]:
rel_paths = 'by_William_Hughes_(only_Bill_Evans)'
fnames = 'Like Someone In Love'


In [142]:
df_raw = preprocess_df(fnames)
df_raw

,mc,mn,mc_onset,mn_onset,timesig,staff,voice,duration,gracenote,nominal_duration,scalar,tied,tpc,midi,volta,chord_id,pc,note,mc_onset_float,order
0,1,1,1/4,1/4,4/4,2,1,3/4,NaN,1/2,3/2,NaN,0,60,NaN,6,0,C,0.250000,1.250000
1,1,1,5/12,5/12,4/4,1,2,1/12,NaN,1/8,2/3,1.0,4,64,NaN,3,4,E,0.416667,1.416667
2,1,1,5/12,5/12,4/4,1,2,1/12,NaN,1/8,2/3,1.0,-1,65,NaN,3,5,F,0.416667,1.416667
3,1,1,5/12,5/12,4/4,1,2,1/12,NaN,1/8,2/3,1.0,3,69,NaN,3,9,A,0.416667,1.416667
4,1,1,5/12,5/12,4/4,1,1,1/12,NaN,1/8,2/3,1.0,0,72,NaN,0,0,C,0.416667,1.416667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3285,171,171,0,0,4/4,1,2,1,NaN,1,1,-1.0,-1,77,NaN,1809,5,F,0.000000,171.000000
3286,171,171,0,0,4/4,1,2,1,NaN,1,1,-1.0,-4,80,NaN,1809,8,G#/Ab,0.000000,171.000000
3287,171,171,0,0,4/4,1,1,1,NaN,1,1,-1.0,-2,82,NaN,1808,10,A#/Bb,0.000000,171.000000
3288,171,171,0,0,4/4,1,1,1,NaN,1,1,-1.0,0,84,NaN,1808,0,C,0.000000,171.000000


In [144]:
pc = generate_pcs(df_raw)
pc

,order,note,pc,pcs
0,1.250000,[C],[0],[0]
1,1.416667,"[E, F, A, C, E, A]","[4, 5, 9, 0, 4, 9]","[0, 9, 4, 5]"
2,1.500000,"[E, F, A, C, E, A]","[4, 5, 9, 0, 4, 9]","[0, 9, 4, 5]"
3,1.750000,[A],[9],[9]
4,2.000000,[C],[0],[0]
...,...,...,...,...
1051,169.750000,"[F, D#/Eb]","[5, 3]","[3, 5]"
1052,169.916667,"[G, G#/Ab, B, D, D#/Eb, G]","[7, 8, 11, 2, 3, 7]","[2, 3, 7, 8, 11]"
1053,170.000000,"[G, G#/Ab, B, D, D#/Eb, G]","[7, 8, 11, 2, 3, 7]","[2, 3, 7, 8, 11]"
1054,170.750000,"[D#/Eb, F, G#/Ab, A#/Bb, C, D#/Eb]","[3, 5, 8, 10, 0, 3]","[0, 3, 5, 8, 10]"


In [93]:
df_raw = get_notes(rel_paths, fnames)
df_raw

,order,measure,beat,part,midi,pc,note,duration
0,1.250000,1,2.000000,MusicXML Part,60,0,C,3.000000
1,1.416748,1,2.666992,MusicXML Part,64,4,E,0.333008
2,1.416748,1,2.666992,MusicXML Part,65,5,F,0.333008
3,1.416748,1,2.666992,MusicXML Part,69,9,A,0.333008
4,1.416748,1,2.666992,MusicXML Part,72,0,C,0.333008
...,...,...,...,...,...,...,...,...
3285,171.000000,171,1.000000,MusicXML Part,77,5,F,4.000000
3286,171.000000,171,1.000000,MusicXML Part,80,8,A-,4.000000
3287,171.000000,171,1.000000,MusicXML Part,82,10,B-,4.000000
3288,171.000000,171,1.000000,MusicXML Part,84,0,C,4.000000


In [147]:
pc = generate_pcs(df_raw)
pc

,order,note,pc,pcs
0,1.250000,[C],[0],[0]
1,1.416667,"[E, F, A, C, E, A]","[4, 5, 9, 0, 4, 9]","[0, 9, 4, 5]"
2,1.500000,"[E, F, A, C, E, A]","[4, 5, 9, 0, 4, 9]","[0, 9, 4, 5]"
3,1.750000,[A],[9],[9]
4,2.000000,[C],[0],[0]
...,...,...,...,...
1051,169.750000,"[F, D#/Eb]","[5, 3]","[3, 5]"
1052,169.916667,"[G, G#/Ab, B, D, D#/Eb, G]","[7, 8, 11, 2, 3, 7]","[2, 3, 7, 8, 11]"
1053,170.000000,"[G, G#/Ab, B, D, D#/Eb, G]","[7, 8, 11, 2, 3, 7]","[2, 3, 7, 8, 11]"
1054,170.750000,"[D#/Eb, F, G#/Ab, A#/Bb, C, D#/Eb]","[3, 5, 8, 10, 0, 3]","[0, 3, 5, 8, 10]"
